In [ ]:
import dotenv
dotenv.load_dotenv()

In [ ]:
import os
from logfire.query_client import LogfireQueryClient

read_token = os.getenv('LOGFIRE_READ_TOKEN')
logfire_query_client = LogfireQueryClient(read_token=read_token)

In [ ]:
trace_rows = logfire_query_client.query_json_rows(
    sql="""
    SELECT
        trace_id,
        start_timestamp,
        duration
    FROM records
    WHERE span_name = 'streamlit_session'
    ORDER BY start_timestamp DESC
    LIMIT 5
    """
)


In [ ]:
trace_ids = [r['trace_id'] for r in trace_rows['rows']]
trace_ids

In [ ]:
trace_id = '019c8f3a23e4cc6bd259a5a00f708b29'

In [ ]:
run_row = logfire_query_client.query_json_rows(
    sql=f"""
    SELECT
        attributes->'pydantic_ai.all_messages' as all_messages,
        attributes->>'gen_ai.usage.input_tokens' as input_tokens,
        attributes->>'gen_ai.usage.output_tokens' as output_tokens
    FROM records
    WHERE trace_id = '{trace_id}'
      AND span_name = 'agent run'
    ORDER BY start_timestamp DESC
    LIMIT 1
    """
)

In [ ]:
all_messages = run_row['rows'][0]

In [ ]:
all_messages['all_messages'][0]

In [ ]:
import sys
sys.path.append('..')

In [ ]:
import trace_replay

In [ ]:
trace = trace_replay.fetch_trace(trace_id, logfire_query_client)

In [ ]:
trace.all_messages[0]

In [ ]:
from models import RAGResponse
run = trace_replay.trace_to_run_result(trace, output_type=RAGResponse)

In [ ]:
run.output.answer

In [ ]:
traces = trace_replay.fetch_traces(trace_ids, logfire_query_client)

In [ ]:
runs = []


for trace in traces.values():
    run = trace_replay.trace_to_run_result(trace)
    runs.append(run)

In [ ]:
import pickle

In [ ]:
with open('../data/logs.bin', 'wb') as f_out:
    pickle.dump(runs, f_out)

In [ ]:
with open('../data/logs.bin', 'rb') as f_in:
    runs = pickle.load(f_in)